# Reuse saved GFL detector and data
Default: restore and inspect only. No dataset downloads or training. Run after the initial preparation has saved the workspace.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, json, os
BASE=Path('/content/drive/MyDrive/Gurnek_GFL_Research')
SAVED=BASE/'workspace'
assert (SAVED/'data/sherbrooke/prepared/annotations/val.json').exists(), 'Initial data cache is not complete yet'
ROOT=Path('/content/deer_research')
shutil.copytree(SAVED, ROOT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('vendor','__pycache__'))
os.chdir(ROOT)
print('Using cached data:',SAVED)


In [ ]:
# Runtime dependencies are temporary; model weights and data stay on Drive.
subprocess.run(['bash','scripts/setup_colab.sh'],check=True)
PY='/content/gfl-env/bin/python'
subprocess.run([PY,'-m','deer_pipeline.relocate'],check=True)
RUNS=BASE/'runs'
completed=RUNS/'complete.json'
if completed.exists():
    MODELS=json.loads(completed.read_text());print(MODELS)
else:
    MODELS={}
    print('Full experiment has not completed. Saved stage checkpoints:')
    print('
'.join(str(p) for p in RUNS.rglob('*.pth')))


## Optional prediction with an existing trained checkpoint
This cell performs inference only. It never retrains. Set `RUN_PREDICTIONS=True` when wanted.


In [ ]:
RUN_PREDICTIONS=False
if RUN_PREDICTIONS:
    assert 'adapted' in MODELS, 'Choose a completed trained checkpoint first'
    weights=Path(MODELS['adapted'])
    subprocess.run([PY,'-m','deer_pipeline','predict',
        '--config',str(weights.parent/'config.py'),
        '--checkpoint',str(weights),
        '--annotations',str(ROOT/'data/decoys/prepared/annotations/test.json'),
        '--output',str(BASE/'reuse_predictions/predictions.json')],check=True)
